# 08-1 Spatial Epidemiology: Floor-Wing Attack Rates and Spot Map

Your supervisor asks: "Where is it worst?"

Pine and Cypress Nursing Home has 3 floors × 2 wings (A / B) and 280 residents in total.
We want to find out which areas have the highest attack rates and draw a spot map.

Workflow: **prepare the data → floor × wing attack rates → heatmap → per-room attack rates → spot map → spatial comparison of case fatality rates**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: Prepare the data ---
import pathlib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

print(f"Residents: {len(df)}")
print(f"Infected: {df['infected'].sum()}")
print(f"Deaths: {df['died'].sum()}")
print(f"\nFloors: {sorted(df['floor'].unique())}")
print(f"Wings: {sorted(df['wing'].unique())}")
print(f"Number of rooms: {df['room'].nunique()}")

### 📌 Reading the Step 1 Results

**Explaining the output**

| Metric | Value | Meaning |
|---|---|---|
| Residents | 280 | All observed subjects in the nursing home |
| Infected | 121 | Number of people with `clinical_severity != "not_ill"` |
| Deaths | 19 | Number of people with `outcome == "dead"` |

**What the flag columns (0/1 indicators) are for**

- `df["infected"]` is 0 (not infected) or 1 (infected) for each person
- Applying `.sum()` to the flag column = number infected; applying `.mean()` = the infection rate (attack rate)
- This is the basic skill of epi calculations, and every later analysis relies on it

> **Tip**: Right after creating a flag, use `print` to check that the total is reasonable (for example, the number infected should never exceed the total number of residents). This is a good habit for avoiding downstream analysis errors.

In [ ]:
# --- Step 2: Floor × Wing attack rates ---
spatial = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
    died=("died", "sum"),
).reset_index()
spatial["attack_rate"] = (spatial["infected"] / spatial["total"] * 100).round(1)
spatial["cfr"] = (spatial["died"] / spatial["infected"] * 100).round(1)

print("=== Floor-wing statistics ===")
print(spatial.to_string(index=False))

### 📌 Step 2 Code Breakdown & Reading the Results

**`groupby().agg()` explained in plain words**

```
spatial = df.groupby(["floor", "wing"])   ← split the 280 rows into 6 groups by floor + wing
    .agg(
        total    = ("case_id",   "count"),  ← total people in each group (count case_id rows)
        infected = ("infected",  "sum"),    ← number infected in each group (sum the flag)
        died     = ("died",      "sum"),    ← number of deaths in each group
    )
    .reset_index()                          ← put floor/wing back as ordinary columns (not the index)
```

**The `agg()` syntax formula**: `new_column_name = ("source_column", "aggregation_function")`

Common aggregation functions: `"count"` (count rows), `"sum"` (add up), `"mean"` (average), `"max"`, `"min"`

**Key points for reading the results**

- `attack_rate`: the higher the value → the harder that wing was hit
- `cfr` (case fatality rate): the proportion of infected people who died, reflecting how severe the illness was
- Comparing attack rate vs. CFR: a high attack rate ≠ a high CFR; look at the two separately

> ⚠️ **Common trap**: Looking only at the absolute case count (like "2F-A has 24 cases, the most!") while ignoring the denominator. 24/44 (54.5%) and 25/50 (50.0%) are close in headcount, but the attack rates barely differ — you can't jump to a conclusion just because a number is large.

In [ ]:
# --- Step 3: Attack rate heatmap ---
heatmap_ar = spatial.pivot(index="floor", columns="wing", values="attack_rate")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Attack rate
sns.heatmap(heatmap_ar, annot=True, fmt=".1f", cmap="YlOrRd",
            cbar_kws={"label": "%"}, ax=axes[0])
axes[0].set_title("Attack Rate (%) by Floor × Wing")
axes[0].set_ylabel("Floor")

# Case fatality rate
heatmap_cfr = spatial.pivot(index="floor", columns="wing", values="cfr")
sns.heatmap(heatmap_cfr, annot=True, fmt=".1f", cmap="Reds",
            cbar_kws={"label": "%"}, ax=axes[1])
axes[1].set_title("Case Fatality Rate (%) by Floor × Wing")
axes[1].set_ylabel("Floor")

plt.tight_layout()
plt.show()

print("→ Areas with the highest attack rates:")
top = spatial.nlargest(3, "attack_rate")
for _, row in top.iterrows():
    print(f"  {row['floor']}F-{row['wing']} wing: {row['attack_rate']}%")

### 📌 Step 3 Code Breakdown & Reading the Heatmap

**What `pivot()` does**

`groupby().agg()` produces a "long table" (each row is one floor-wing combination):
```
floor  wing  attack_rate
1      A     34.1
1      B     21.3
...
```

`pivot(index="floor", columns="wing", values="attack_rate")` turns it into a "matrix":
```
wing    A     B
floor
1      34.1  21.3
2      54.5  50.0
3      41.7  57.4
```

`sns.heatmap()` needs matrix format to draw the cells, so **`pivot()` is a required preprocessing step**.

**`heatmap` parameters explained**

| Parameter | Meaning |
|---|---|
| `annot=True` | Display the value inside each cell (otherwise you only get color) |
| `fmt=".1f"` | Number format: one decimal place |
| `cmap="YlOrRd"` | Palette: yellow → orange → red, intuitively matching low → medium → high attack rate |
| `cmap="Reds"` | Use a pure-red gradient for the CFR to visually distinguish it from the attack rate chart |

**Reading the heatmap**

- The darker (redder) the color = the higher the attack rate / CFR
- Look across a **row** (same floor, wing A vs. B): is one wing systematically higher?
- Look down a **column** (same wing, different floors): are 2F/3F worse than 1F?
- A wing with both a high attack rate and a high CFR → that area may have an especially dangerous exposure

> **When do you use a heatmap?** When your subject happens to have **two categorical dimensions** (floor + wing), a heatmap is the most intuitive choice. With three dimensions, you'd need to consider faceting or an interactive chart.

In [ ]:
# --- Step 4: Attack rate per room ---
room_stats = df.groupby("room").agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
).reset_index()
room_stats["attack_rate"] = (room_stats["infected"] / room_stats["total"] * 100).round(1)

# Parse the room name (e.g. "2A-03" → floor=2, wing=A, room_num=3)
room_stats["floor_num"] = room_stats["room"].str[0].astype(int)
room_stats["wing_code"] = room_stats["room"].str[1]
room_stats["room_num"] = room_stats["room"].str.split("-").str[1].astype(int)

print(f"{len(room_stats)} rooms in total")
print(f"\nRooms with a 100% attack rate (every resident infected):")
full = room_stats[room_stats["attack_rate"] == 100.0]
print(f"  {len(full)} rooms")
print(f"\nRooms with a 0% attack rate (no one infected):")
zero = room_stats[room_stats["attack_rate"] == 0.0]
print(f"  {len(zero)} rooms")

### 📌 Step 4 Code Breakdown & Reading the Results

**String parsing broken down**

Room code format: `"2A-03"` (floor + wing code + hyphen + room number)

| Operation | Using "2A-03" | Explanation |
|---|---|---|
| `room.str[0].astype(int)` | `"2"` → `2` | Take the 0th character = floor number |
| `room.str[1]` | `"A"` | Take the 1st character = wing code |
| `room.str.split("-").str[1].astype(int)` | `"03"` → `3` | Split on `-`, take the second part = room number |

**Why parse it into numbers?**

`scatter()`'s x/y need **numeric** values to place coordinates. The string `"2A-03"` can't be placed on a number line directly; the parsed integers `2` and `3` can.

**Reading the results**

- Rooms with a 100% attack rate: every resident in the room was infected → but note, are these single-occupancy or multi-occupancy rooms?
  - 1 infected out of 1 = 100% has a completely different epidemiological meaning than 5 infected out of 5 = 100%
- Rooms with a 0% attack rate: everyone was spared → is there a protective factor? (e.g. not using shared hot-water facilities)

> The attack rate of small rooms (1-2 people) is very unstable; 100% may just be a coincidental 1/1 and shouldn't be over-interpreted. The dot sizes in the Step 5 spot map exist to visualize this very problem.

In [ ]:
# --- Step 5: Spot Map (mimicking the nursing home floor plan) ---
# X axis = room number, wing A on the left half, wing B on the right half
# Y axis = floor
max_room_a = room_stats[room_stats["wing_code"] == "A"]["room_num"].max()
gap = 5  # gap between wing A and wing B

room_stats["x"] = room_stats.apply(
    lambda r: r["room_num"] if r["wing_code"] == "A"
    else r["room_num"] + max_room_a + gap,
    axis=1,
)

fig, ax = plt.subplots(figsize=(14, 5))
sc = ax.scatter(
    room_stats["x"],
    room_stats["floor_num"],
    s=room_stats["total"] * 50,
    c=room_stats["attack_rate"],
    cmap="YlOrRd",
    edgecolors="black",
    linewidth=0.5,
    alpha=0.8,
    vmin=0,
    vmax=100,
)
plt.colorbar(sc, label="Attack Rate (%)")

# Divider line and labels
mid_x = max_room_a + gap / 2
ax.axvline(x=mid_x, color="gray", linestyle="--", alpha=0.5)
ax.text(max_room_a / 2, 3.5, "Wing A", ha="center", fontsize=12, fontweight="bold")
ax.text(max_room_a + gap + 12, 3.5, "Wing B", ha="center", fontsize=12, fontweight="bold")

ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["1F", "2F", "3F"])
ax.set_xlabel("Room number")
ax.set_ylabel("Floor")
ax.set_title("Spot Map — attack rate per room (dot size = residents, color = attack rate)")
plt.tight_layout()
plt.show()

print("→ Are the dark (high-attack-rate) dots concentrated in certain wings?")
print("→ These high-risk areas may share contaminated hot-water piping or showerheads")

### 📌 Step 5 Code Breakdown & Reading the Spot Map

**The x-coordinate offset design**

Both wing A and wing B number their rooms starting from 01. If you use `room_num` directly as the x coordinate, the dots from the two wings will overlap.

The fix:
```
wing B's x = wing B room number + wing A's maximum room number + gap(=5)
```

For example, if wing A has 15 rooms (room_num 1~15), then wing B's room 1 has x = 1 + 15 + 5 = 21.
This way wing A occupies x ∈ [1,15], the gap sits in [16,20], and wing B starts at x=21, mimicking the left-right layout of a real floor plan.

> Using `max_room_a + gap` instead of hard-coding 30 means that if the data changes (the number of rooms in wing A changes), the code adjusts automatically without any manual editing.

**scatter parameters explained**

| Parameter | What it represents |
|---|---|
| `s = total * 50` | Dot **area** (s = size); multiplying by 50 is a visual scaling factor |
| `c = attack_rate` | Dot color corresponds to the attack rate |
| `cmap="YlOrRd"` | Same palette as the heatmap, keeping the visuals consistent |
| `vmin=0, vmax=100` | Fix the color-axis range to 0-100% (otherwise different charts have inconsistent color baselines) |
| `alpha=0.8` | Transparency of 0.8, keeping overlapping dots visible |

**Reading the spot map (four questions)**

1. **Where is it reddest?** → clusters of dark dots = groups of high-risk rooms; investigate the water source first
2. **Big dots vs. small dots?** → the attack rate of big dots is more trustworthy; small dots (a 100% single-occupancy room) are just chance and shouldn't be over-interpreted
3. **Wing A vs. wing B** → is there a big color difference across the divider line? → wing-specific exposure (different piping?)
4. **Floor level** → are 2F/3F generally darker than 1F? → the vertical piping may have a problem

> **Heatmap vs. spot map — which one?**
> - Heatmap: gives an overall summary of the 6 floor×wing combinations, good for reports and decision-making
> - Spot map: shows the detailed distribution of each room and finds anomalous clusters, good for in-depth investigation

In [ ]:
# --- Step 6: Sorted bar chart of wing attack rates ---
spatial["label"] = spatial["floor"].astype(str) + "F-" + spatial["wing"]
spatial_sorted = spatial.sort_values("attack_rate", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(
    spatial_sorted["label"],
    spatial_sorted["attack_rate"],
    color=["#e34a33" if ar > 50 else "#2c7fb8" for ar in spatial_sorted["attack_rate"]],
)
for bar, val in zip(bars, spatial_sorted["attack_rate"]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f"{val}%", va="center")

ax.set_xlabel("Attack Rate (%)")
ax.set_title("Attack Rate by Wing (red > 50%)")
ax.set_xlim(0, 70)
plt.tight_layout()
plt.show()

print("→ Wings with an attack rate above 50% need environmental sampling first")

### 📌 Step 6 Interpretation & Next Steps

**Reading the bar chart**

- The longer the bar and the redder the color → the higher the attack rate and the more urgent the action
- Red (>50%) means **more than half the residents were infected**, a strong warning sign
- Sorting lets you see the relative risk at a glance: the wing furthest to the right is the most dangerous

**After spotting a difference between wings, the next steps**

| Question | Analysis method |
|---|---|
| Where is mortality also high? | Compare with the CFR heatmap on the left |
| Is shower usage concentrated in the high-attack-rate wings? | `groupby("wing").agg(shower_pct=("shower_use","mean"))` |
| What infrastructure do the high-risk areas share? | Cross-reference the nursing home engineering plans and mark the hot-water piping routes |
| Is there statistical significance to the spatial clustering? | Advanced: Moran's I spatial autocorrelation (beyond this chapter) |

**A template conclusion sentence**

> "Wing 3F-B has the highest attack rate (57.4%), followed by 2F-A (54.5%) and 2F-B (50.0%). These three wings should be prioritized for environmental sampling of the hot-water system, and their shared piping should be investigated."

## Summary

| Step | Skill learned |
|------|------------|
| Floor × wing attack rates | Multi-metric computation with `groupby().agg()` |
| Heatmap | `sns.heatmap()` + `pivot()` |
| Per-room analysis | Parsing the `room` string column |
| Spot map | `scatter()` with size = residents, color = attack rate |
| Sorted bar chart | Marking high-risk areas with color |

**Conclusion**: 2F and wing 3F-B have the highest attack rates; these areas may share a contaminated hot-water system.
The next notebook (`08_spatial_choropleth`) demonstrates the concept of a geographic choropleth.